# Evidencia de ingesta on-chain

> Cuaderno de validación reproducible del pipeline ETL on-chain sobre las tablas `onchain_data.fact_onchain_*_4h`, snapshots `parquet` validados y artefactos exportables para la memoria del TFG.

## Contexto y motivación

La capa *on-chain* aporta una señal distinta a precio y microestructura: intenta medir actividad económica propia de cada red, no solo lo que ocurre en el libro de órdenes. En este proyecto se usa como bloque explicativo para cinco activos (`BTC`, `ETH`, `BNB`, `XRP`, `SOL`) agregados en buckets de 4 horas, la misma rejilla temporal 4h que la ingesta OHLCV (`02_evidencia_ingesta_ohlcv.ipynb`, `notebooks/01_ingesta/ohlcv/`) y el dataset supervisado aguas abajo.

Validar esta fuente es especialmente importante por dos razones. Primero, cada activo procede de consultas y transformaciones específicas, por lo que no basta con comprobar que existe una tabla genérica. Segundo, las series on-chain pueden tener arranques históricos distintos y algunos rellenos de rejilla: si eso no queda medido, el modelo podría interpretar como información real lo que en realidad es una fila sintética para mantener la continuidad temporal.

Este cuaderno convierte la validación técnica en una evidencia legible:

- ejecuta el validador **end-to-end** de on-chain, con el registro técnico capturado fuera de la salida visible;
- resume cobertura temporal por activo y tabla fact;
- revisa auditoría de runs históricos, live y repair cuando existe en BD;
- comprueba huecos en la serie, duplicados e integridad real frente a relleno de rejilla (`is_grid_padded`);
- compila en tabla el inventario de los snapshots `parquet` canónicos que respaldan la evidencia SQL;
- exporta JSON/CSV versionados en `reports/validation/onchain/`.

El flujo está pensado para `Kernel > Restart Kernel and Run All` y sigue el mismo orden que las evidencias de otras ETLs.

## 1. Configuración del entorno

Se localiza la raíz del proyecto buscando hacia arriba desde el `cwd` actual hasta encontrar `scripts/validate_onchain_pipeline.py`. Así el cuaderno puede ejecutarse desde `notebooks/01_ingesta/onchain/`, desde la raíz del repositorio o desde el contenedor Docker (raíz `/app`).

También se guardan metadatos mínimos de ejecución (`fecha_hora_utc`, raíz, versión de Python y plataforma). No cambian la validación, pero hacen que los CSV/JSON exportados sean trazables si se revisan semanas después.

In [1]:
from __future__ import annotations

import contextlib
import importlib
import io
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from sqlalchemy import text

# Fija la raíz del repositorio, las constantes on-chain, la congelación y un `RUN_STAMP` compartido
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "scripts" / "validate_onchain_pipeline.py").exists():
            return candidate
    raise RuntimeError("No se encontró la raíz del proyecto (falta scripts/validate_onchain_pipeline.py).")


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import scripts.validate_onchain_pipeline as val
from src.config.settings import Settings
from src.utils.database import create_db_engine

val = importlib.reload(val)  # Recarga el módulo y alinea la caché del kernel con el disco

SETTINGS_PATH = ROOT / "config" / "settings.yaml"
ONCHAIN_SCHEMA = "onchain_data"
ONCHAIN_TABLES = (
    "fact_onchain_btc_4h",
    "fact_onchain_eth_4h",
    "fact_onchain_sol_4h",
    "fact_onchain_xrp_4h",
    "fact_onchain_bnb_4h",
)
ONCHAIN_TABLE_TO_COIN = {
    "fact_onchain_btc_4h": "BTC",
    "fact_onchain_eth_4h": "ETH",
    "fact_onchain_sol_4h": "SOL",
    "fact_onchain_xrp_4h": "XRP",
    "fact_onchain_bnb_4h": "BNB",
}
ONCHAIN_NET_FLOW_COL = {
    table: f"net_flow_{coin.lower()}"
    for table, coin in ONCHAIN_TABLE_TO_COIN.items()
}
ONCHAIN_ASSETS = tuple(ONCHAIN_TABLE_TO_COIN.values())
ONCHAIN_PARQUET_EXPECTED_FILES = len(ONCHAIN_TABLES)
ONCHAIN_COVERAGE_COLUMNS = ["simbolo", "filas", "min_timestamp", "max_timestamp"]
ONCHAIN_PARQUET_COLUMNS = [
    "tipo",
    "simbolo",
    "archivo",
    "tamaño_mb",
    "filas",
    "columnas",
    "min_timestamp",
    "max_timestamp",
]
ONCHAIN_RAW_ROOT = ROOT / "data" / "01_raw" / "onchain"
ONCHAIN_OUTPUT_DIR = ROOT / "reports" / "validation" / "onchain"
RUN_TIMESTAMP = datetime.now(timezone.utc)
RUN_STAMP = RUN_TIMESTAMP.strftime("%Y%m%dT%H%M%SZ")

settings = Settings.load_from_yaml(SETTINGS_PATH)
ONCHAIN_FREEZE_COVERAGE_DATE = settings.trading_universe.freeze_date
ONCHAIN_FREEZE_KWARGS = {"target_date": ONCHAIN_FREEZE_COVERAGE_DATE}
ONCHAIN_FREEZE_BUCKET = (
    pd.Timestamp(val.freeze_target_bucket(ONCHAIN_FREEZE_COVERAGE_DATE))
    if ONCHAIN_FREEZE_COVERAGE_DATE
    else None
)

run_meta = {
    "fecha_hora_utc": RUN_TIMESTAMP.isoformat(),
    "raiz_proyecto": str(ROOT),
    "python": platform.python_version(),
    "plataforma": platform.platform(),
}
run_meta

{'fecha_hora_utc': '2026-06-04T00:08:28.927380+00:00',
 'raiz_proyecto': '/app',
 'python': '3.10.19',
 'plataforma': 'Linux-5.15.133.1-microsoft-standard-WSL2-x86_64-with-glibc2.36'}

## 2. Ejecución del validador end-to-end

Esta celda ejecuta `scripts/validate_onchain_pipeline.py` desde el cuaderno.

La matriz `summary_df` resume las comprobaciones principales: conexión a SQL, disponibilidad de las cinco tablas de facts, baterías de pruebas, integración con BD, calidad mínima y cobertura hasta el freeze. Para leerla, basta con mirar `estado` y `requerido`: un `FAIL` requerido bloquea la aceptación; un `SKIP` explica una precondición que no se pudo comprobar en ese entorno.

In [2]:
# Ejecuta el E2E, rellena `val.results` y forma `summary_df` con el resumen de aceptación
val.results.clear()

# Desvía trazas técnicas; deja la salida visible al mínimo en esta celda
validator_logs_buffer = io.StringIO()
with contextlib.redirect_stdout(validator_logs_buffer), contextlib.redirect_stderr(validator_logs_buffer):
    db_ready = val.check_db_connection()
    val.check_onchain_tables()
    val.run_unit_suite()

    if db_ready and val.results.get("TABLES_READY") == val.PASS:
        val.run_db_integration_suite()
        val.check_data_quality()
        val.check_freeze_coverage(**ONCHAIN_FREEZE_KWARGS)
    else:
        val.results["DB_INTEGRATION"] = val.SKIP
        val.set_status(
            val.results,
            check_id="DATA_QUALITY_MIN",
            status=val.SKIP,
            aliases=("DATA_QUALITY",),
        )
        val.results["FREEZE_COVERAGE"] = val.SKIP

    val.check_audit_evidence(db_integration_passed=val.results.get("DB_INTEGRATION") == val.PASS)

    acceptance_matrix = val._build_onchain_acceptance_matrix(
        **ONCHAIN_FREEZE_KWARGS,
        db_ready=db_ready,
        onchain_tables_ready=val.results.get("TABLES_READY") == val.PASS,
    )
    accepted, actionable_failures = val._print_summary(acceptance_matrix)

logs_tecnicos = validator_logs_buffer.getvalue()
print("Ejecución completada. Registro técnico en la variable logs_tecnicos (oculto en la salida de la celda).")

matrix_required = {str(row["check_id"]): bool(row["required"]) for row in acceptance_matrix}


def detalle_check(check_id: str, status: str) -> str:
    if status == val.FAIL:
        for failure in actionable_failures:
            if check_id in failure:
                return failure
        return "Fallo detectado. Revisar logs de validación para diagnóstico."

    if status == val.SKIP:
        if check_id == "FREEZE_COVERAGE" and not ONCHAIN_FREEZE_COVERAGE_DATE:
            return "No aplica en esta ejecución (sin fecha de freeze de política)."
        return "No ejecutado por precondiciones no cumplidas."

    detalles_ok = {
        "INTEGRATION": "Suite unitaria on-chain ejecutada correctamente.",
        "DB_INTEGRATION": "Suite db_integration on-chain ejecutada correctamente.",
        "AUDIT_RUNS": "Evidencia de auditoría (cubierta por tests DB si aplica).",
        "AUDIT_EVENTS": "Evidencia de auditoría (cubierta por tests DB si aplica).",
        "DATA_QUALITY_MIN": (
            "Sin duplicados ni dominio net_flow inválido. "
            "La alineación temporal al freeze la cubre FREEZE_COVERAGE."
        ),
        "FREEZE_COVERAGE": (
            "Cobertura temporal validada hasta el bucket freeze 4h."
            if ONCHAIN_FREEZE_COVERAGE_DATE
            else "No requerido en esta ejecución."
        ),
    }
    return detalles_ok.get(check_id, "Comprobación validada correctamente.")


summary_df = pd.DataFrame(
    [
        {
            "id_check": check_id,
            "etiqueta": label,
            "estado": status,
            "requerido": "Sí" if matrix_required.get(check_id, False) else "No",
            "detalle": detalle_check(check_id, status),
        }
        for check_id, label in val._CHECK_LABELS.items()
        for status in [val.results.get(check_id, val.SKIP)]
    ]
)
summary_df

Ejecución completada. Registro técnico en la variable logs_tecnicos (oculto en la salida de la celda).


,id_check,etiqueta,estado,requerido,detalle
0,DB_CONNECTION,Conexión a TimescaleDB,PASS,Sí,Comprobación validada correctamente.
1,TABLES_READY,Tablas onchain_data (5 fact) disponibles,PASS,Sí,Comprobación validada correctamente.
2,INTEGRATION,"Suite unitaria on-chain (loaders, merger, jobs...",PASS,Sí,Suite unitaria on-chain ejecutada correctamente.
3,DB_INTEGRATION,Suite de integración con BD on-chain,PASS,Sí,Suite db_integration on-chain ejecutada correc...
4,AUDIT_RUNS,Evidencia de auditoría (cubierta por tests DB ...,PASS,No,Evidencia de auditoría (cubierta por tests DB ...
5,AUDIT_EVENTS,Evidencia de auditoría (cubierta por tests DB ...,PASS,No,Evidencia de auditoría (cubierta por tests DB ...
6,DATA_QUALITY_MIN,Deduplicación PK + dominio net_flow,PASS,Sí,Sin duplicados ni dominio net_flow inválido. L...
7,FREEZE_COVERAGE,Cobertura por activo/tabla hasta bucket objeti...,PASS,Sí,Cobertura temporal validada hasta el bucket fr...


### 2.1. Lectura de la matriz de aceptación

Las ocho comprobaciones devueltas cubren las capas críticas del ETL on-chain:

| Capa | Comprobaciones asociadas | Por qué importa |
|---|---|---|
| **Infraestructura** | `DB_CONNECTION`, `TABLES_READY` | Sin BD viva ni las cinco facts (`fact_onchain_*_4h`) creadas, no hay nada que validar |
| **Pipeline** | `INTEGRATION` | Confirma que loaders, merger, jobs y consultas Dune ejecutan la suite unitaria sin excepciones |
| **Datos persistidos** | `DB_INTEGRATION`, `DATA_QUALITY_MIN` | Hay filas en cada fact, cero duplicados por `bucket_4h` y dominio válido de `net_flow` |
| **Auditoría** | `AUDIT_RUNS`, `AUDIT_EVENTS` | Trazabilidad en `onchain_data.ingestion_runs` cuando el esquema la expone; en este validador son **opcionales** (equivalente cubierto por la suite de integración en BD) |
| **Cobertura temporal** | `FREEZE_COVERAGE` | Para cada activo/tabla, `max(bucket_4h) >=` **2026-02-28 20:00 UTC** (*freeze* de política); datos posteriores o iguales a ese bucket -> PASS |

**En esta ejecución** (`stamp` `20260603T235232Z`, `fecha_hora_utc` **2026-06-03 23:52 UTC**), las ocho comprobaciones de `summary_df` figuran en `PASS` y el export cierra con resultado global `PASS`. Eso valida la **base mínima** sobre la que puede integrarse la señal *on-chain* en el backbone multifuente. Si alguna apareciera en `FAIL`, la columna `detalle` apunta a la pista accionable (tabla vacía, duplicado por bucket, cobertura por debajo del *freeze*, etc.); un `SKIP` indica precondiciones no cumplidas (por ejemplo, BD no disponible o facts no listas).


## 3.1. Cobertura SQL por tabla

La primera lectura baja al dato persistido: cuántas filas tiene cada fact y cuál es su ventana temporal real. Esta tabla explica bastante de la naturaleza de la fuente: BTC y ETH arrancan antes, mientras que SOL y BNB tienen históricos más cortos por disponibilidad de red/fuente.

In [3]:
# Conecta, mide cobertura 4h por hecho y resume el `net_flow` no nulo
db_logs_buffer = io.StringIO()
with contextlib.redirect_stdout(db_logs_buffer), contextlib.redirect_stderr(db_logs_buffer):
    engine = create_db_engine()
logs_conexion_sql = db_logs_buffer.getvalue()

coverage_frames: list[pd.DataFrame] = []
stats_dfs: dict[str, pd.DataFrame] = {}

if engine:
    for table in ONCHAIN_TABLES:
        coverage_query = text(
            f"""
            SELECT COUNT(*) AS filas,
                   MIN(bucket_4h) AS min_timestamp,
                   MAX(bucket_4h) AS max_timestamp
            FROM {ONCHAIN_SCHEMA}.{table}
            """
        )
        coverage_frame = pd.read_sql(coverage_query, engine)
        coverage_frame.insert(0, "tabla", table)
        coverage_frame.insert(1, "simbolo", ONCHAIN_TABLE_TO_COIN[table])
        coverage_frames.append(coverage_frame)

        net_flow_column = ONCHAIN_NET_FLOW_COL[table]
        distribution_query = text(
            f"SELECT {net_flow_column} AS net_flow FROM {ONCHAIN_SCHEMA}.{table} "
            f"WHERE {net_flow_column} IS NOT NULL"
        )
        stats_dfs[table] = pd.read_sql(distribution_query, engine).describe()

evidencia_cobertura_df = (
    pd.concat(coverage_frames, ignore_index=True)
    if coverage_frames
    else pd.DataFrame()
)
evidencia_cobertura_vista_df = (
    evidencia_cobertura_df.assign(
        tabla=lambda data: ONCHAIN_SCHEMA + "." + data["tabla"].astype(str),
    )[["simbolo", "tabla", "filas", "min_timestamp", "max_timestamp"]]
    if not evidencia_cobertura_df.empty
    else pd.DataFrame(columns=["simbolo", "tabla", "filas", "min_timestamp", "max_timestamp"])
)

coverage_source_df = (
    evidencia_cobertura_df[ONCHAIN_COVERAGE_COLUMNS]
    if not evidencia_cobertura_df.empty
    else pd.DataFrame(columns=ONCHAIN_COVERAGE_COLUMNS)
)
cobertura_objetivo_df = pd.DataFrame({"simbolo": ONCHAIN_ASSETS}).merge(
    coverage_source_df,
    on="simbolo",
    how="left",
)
cobertura_objetivo_df["filas"] = cobertura_objetivo_df["filas"].fillna(0).astype(int)
cobertura_objetivo_df["tiene_datos"] = cobertura_objetivo_df["filas"] > 0

if ONCHAIN_FREEZE_BUCKET is not None and not cobertura_objetivo_df.empty:
    cobertura_objetivo_df["max_timestamp"] = pd.to_datetime(
        cobertura_objetivo_df["max_timestamp"], utc=True, errors="coerce"
    )
    cobertura_objetivo_df["max_timestamp"] = cobertura_objetivo_df["max_timestamp"].where(
        cobertura_objetivo_df["max_timestamp"].isna()
        | (cobertura_objetivo_df["max_timestamp"] <= ONCHAIN_FREEZE_BUCKET),
        ONCHAIN_FREEZE_BUCKET,
    )

evidencia_cobertura_vista_df

,simbolo,tabla,filas,min_timestamp,max_timestamp
0,BTC,onchain_data.fact_onchain_btc_4h,19131,2017-08-01 00:00:00+00:00,2026-04-24 08:00:00+00:00
1,ETH,onchain_data.fact_onchain_eth_4h,19130,2017-08-01 00:00:00+00:00,2026-04-24 04:00:00+00:00
2,SOL,onchain_data.fact_onchain_sol_4h,12174,2020-10-03 12:00:00+00:00,2026-04-24 08:00:00+00:00
3,XRP,onchain_data.fact_onchain_xrp_4h,17458,2018-05-01 00:00:00+00:00,2026-04-18 12:00:00+00:00
4,BNB,onchain_data.fact_onchain_bnb_4h,12369,2020-09-01 00:00:00+00:00,2026-04-24 08:00:00+00:00


## 3.2. Auditoría de ejecuciones histórico/live/repair

Tras comprobar que las facts existen, se contrasta la trazabilidad operativa. La tabla `onchain_data.ingestion_runs`, cuando está disponible, permite separar corridas históricas, live y repair, contar fallos y ver la última ejecución registrada.

Esta parte no sustituye las comprobaciones de calidad sobre las facts: sirve para explicar cómo se llegó al estado actual de los datos y para detectar si una fuente se ha mantenido solo por backfill, por live o por reparaciones posteriores.

In [4]:
# Mapea `pipeline_type` a una categoría, lee runs y agrega resúmenes por backfill, live o repair
def modo_pipeline(pipeline_type: object) -> str:
    pipeline_name = str(pipeline_type).lower()
    if "live" in pipeline_name:
        return "live"
    if "repair" in pipeline_name:
        return "repair"
    if "backfill" in pipeline_name:
        return "histórico"
    return "otro"


# Lee onchain_data.ingestion_runs (hasta 200) o devuelve un `DataFrame` vacío sin motor
def leer_auditoria_onchain() -> pd.DataFrame:
    if not engine:
        return pd.DataFrame()

    audit_query = text(
        f"""
        SELECT
            r.run_id,
            r.pipeline_type,
            r.started_at,
            r.finished_at,
            r.success_count,
            r.failed_count,
            r.total_rows
        FROM {ONCHAIN_SCHEMA}.ingestion_runs r
        WHERE r.pipeline_type IN (
            'etl_onchain_backfill',
            'etl_onchain_live',
            'etl_onchain_repair'
        )
        ORDER BY r.started_at DESC
        LIMIT 200;
        """
    )
    with engine.connect() as connection:
        return pd.read_sql(audit_query, connection)



# Agrega por modo los conteos, filas y fracción de runs con al menos un fallo asociado al evento
def resumir_modos_onchain(audit_df: pd.DataFrame) -> pd.DataFrame:
    if audit_df.empty:
        return pd.DataFrame()

    audit_df["modo"] = audit_df["pipeline_type"].map(modo_pipeline)
    audit_df["run_con_fallo"] = audit_df["failed_count"].fillna(0).astype(int) > 0
    mode_summary_df = (
        audit_df.groupby("modo", as_index=False)
        .agg(
            ejecuciones=("run_id", "count"),
            ultimo_inicio=("started_at", "max"),
            total_filas=("total_rows", "sum"),
            total_fallos_eventos=("failed_count", "sum"),
            ejecuciones_con_fallo=("run_con_fallo", "sum"),
        )
        .sort_values("modo")
    )
    mode_summary_df["pct_ejecuciones_con_fallo"] = (
        (mode_summary_df["ejecuciones_con_fallo"] / mode_summary_df["ejecuciones"]) * 100.0
    ).round(2)
    return mode_summary_df


evidencia_auditoria_onchain_df = leer_auditoria_onchain()
resumen_modos_onchain_df = resumir_modos_onchain(evidencia_auditoria_onchain_df)

print("Resumen de modos de ejecución on-chain")
display(resumen_modos_onchain_df)

Resumen de modos de ejecución on-chain


,modo,ejecuciones,ultimo_inicio,total_filas,total_fallos_eventos,ejecuciones_con_fallo,pct_ejecuciones_con_fallo
0,histórico,1,2026-04-18 14:22:22.841158+00:00,17166,0,0,0.0
1,live,16,2026-04-24 12:16:38.140586+00:00,146428,0,0,0.0


## 3.3. Detección de huecos temporales

Con la cobertura agregada no basta: una serie puede empezar y terminar en fechas correctas, pero tener huecos intermedios. Aquí se compara cada `bucket_4h` con el anterior dentro de cada fact y se reporta cualquier salto superior a 4 horas.

Si no aparece ninguna tabla, la interpretación es sencilla: la rejilla temporal está cerrada para todas las series on-chain disponibles.

In [5]:
# Lista los saltos estrictos de más de 4 h entre `bucket_4h` consecutivos
gaps_dfs: dict[str, pd.DataFrame] = {}

if engine:
    for table in ONCHAIN_TABLES:
        gaps_query = text(
            f"""
            WITH ordered AS (
                SELECT bucket_4h,
                       LAG(bucket_4h) OVER (ORDER BY bucket_4h) AS prev_bucket
                FROM {ONCHAIN_SCHEMA}.{table}
            )
            SELECT prev_bucket AS gap_desde,
                   bucket_4h   AS gap_hasta,
                   EXTRACT(EPOCH FROM (bucket_4h - prev_bucket)) / 3600 AS horas_gap
            FROM ordered
            WHERE bucket_4h - prev_bucket > INTERVAL '4 hours'
            ORDER BY gap_desde;
            """
        )
        gaps_frame = pd.read_sql(gaps_query, engine)
        if not gaps_frame.empty:
            gaps_dfs[table] = gaps_frame

if gaps_dfs:
    for table, gaps_frame in gaps_dfs.items():
        print(f"\n--- Huecos en {table} ({ONCHAIN_TABLE_TO_COIN[table]}): {len(gaps_frame)} salto(s) estricto(s) > 4h ---")
        display(gaps_frame)
else:
    print("No se detectaron huecos en ninguna tabla on-chain.")

No se detectaron huecos en ninguna tabla on-chain.


## 3.4. Duplicados por clave temporal

Cada activo vive en su propia fact table, así que la unicidad práctica se reduce a no repetir `bucket_4h` dentro de cada tabla. Aun así, se añaden `simbolo` y `timeframe` en la vista para conservar el formato de las evidencias de otros cuadernos.

El valor esperado de `duplicados_pk` es 0 para todos los activos.

In [6]:
# Compara filas totales y claves 4h distintas en cada hecho on-chain
duplicate_frames: list[pd.DataFrame] = []

if engine:
    for table in ONCHAIN_TABLES:
        duplicates_query = text(
            f"""
            SELECT
                COUNT(*) AS total_filas,
                COUNT(DISTINCT bucket_4h) AS timestamps_unicos,
                COUNT(*) - COUNT(DISTINCT bucket_4h) AS duplicados_pk
            FROM {ONCHAIN_SCHEMA}.{table}
            """
        )
        duplicate_frame = pd.read_sql(duplicates_query, engine)
        duplicate_frame.insert(0, "simbolo", ONCHAIN_TABLE_TO_COIN[table])
        duplicate_frame.insert(1, "timeframe", "4h")
        duplicate_frames.append(duplicate_frame)

duplicados_pk_df = (
    pd.concat(duplicate_frames, ignore_index=True)
    if duplicate_frames
    else pd.DataFrame()
)
duplicados_pk_df

,simbolo,timeframe,total_filas,timestamps_unicos,duplicados_pk
0,BTC,4h,19131,19131,0
1,ETH,4h,19130,19130,0
2,SOL,4h,12174,12174,0
3,XRP,4h,17458,17458,0
4,BNB,4h,12369,12369,0


## 3.5. Integridad real frente a relleno de rejilla

Esta es la parte más específica de on-chain. En OHLCV se habla de imputaciones con `is_imputed`; aquí el equivalente conceptual es `is_grid_padded`, que marca filas sintéticas añadidas para mantener una rejilla 4h densa cuando la fuente no trae una observación real.

La columna `pct_integridad_real` mide la proporción de filas no rellenadas. **En esta corrida**, BTC, ETH y XRP muestran **100 %** de integridad real (`filas_imputadas = 0`); SOL registra **18** filas `is_grid_padded` (**99.85 %** real) y BNB **1** (**99.99 %**). Eso no invalida la fuente, pero documenta qué parte de la continuidad temporal es densificación del ETL y no dato observado.

In [7]:
# Mide relleno de rejilla vía `is_grid_padded` y `pct_integridad_real` por hecho on-chain
integrity_frames: list[pd.DataFrame] = []

if engine:
    padded_column_query = text(
        """
        SELECT 1
        FROM information_schema.columns
        WHERE table_schema = :schema
          AND table_name = :table
          AND column_name = 'is_grid_padded'
        LIMIT 1
        """
    )
    has_padded_column = bool(
        pd.read_sql(
            padded_column_query,
            engine,
            params={"schema": ONCHAIN_SCHEMA, "table": ONCHAIN_TABLES[0]},
        ).shape[0]
    )
    if has_padded_column:
        for table in ONCHAIN_TABLES:
            integrity_query = text(
                f"""
                SELECT
                    COUNT(*) AS total_filas,
                    SUM(CASE WHEN is_grid_padded THEN 1 ELSE 0 END) AS filas_imputadas,
                    SUM(CASE WHEN NOT is_grid_padded THEN 1 ELSE 0 END) AS filas_reales,
                    ROUND(
                        100.0 * SUM(CASE WHEN NOT is_grid_padded THEN 1 ELSE 0 END)
                        / NULLIF(COUNT(*), 0), 4
                    ) AS pct_integridad_real
                FROM {ONCHAIN_SCHEMA}.{table}
                """
            )
            integrity_frame = pd.read_sql(integrity_query, engine)
            integrity_frame.insert(0, "simbolo", ONCHAIN_TABLE_TO_COIN[table])
            integrity_frames.append(integrity_frame)

integridad_onchain_df = (
    pd.concat(integrity_frames, ignore_index=True)
    if integrity_frames
    else pd.DataFrame()
)
integridad_onchain_ok = bool(
    not integridad_onchain_df.empty and len(integridad_onchain_df) == len(ONCHAIN_TABLES)
)

print("Integridad e imputaciones por activo")
display(integridad_onchain_df)
print(f"Integridad por activo: {'OK' if integridad_onchain_ok else 'PENDIENTE'}")

Integridad e imputaciones por activo


,simbolo,total_filas,filas_imputadas,filas_reales,pct_integridad_real
0,BTC,19131,0,19131,100.0000
1,ETH,19130,0,19130,100.0000
2,SOL,12174,18,12156,99.8521
3,XRP,17458,0,17458,100.0000
4,BNB,12369,1,12368,99.9919


Integridad por activo: OK


## 4. Inventario de artefactos Parquet en disco

La BD es la fuente consultable, pero los snapshots `parquet` son el respaldo reproducible de esta capa. El ETL materializa un fichero `fact_onchain_<activo>_4h_validated.parquet` por activo bajo `data/01_raw/onchain/<activo>/`.

La tabla siguiente comprueba que existen los cinco ficheros esperados y resume tamaño, número de filas, número de columnas y ventana temporal. Si un fichero faltara, la fila quedaría igualmente registrada con el nombre esperado para que el problema sea visible.

In [8]:
# Recorre el snapshot canónico e incluye en el inventario los `*_validated.parquet` bajo el árbol on-chain
parquet_rows: list[dict] = []

for table in ONCHAIN_TABLES:
    coin = ONCHAIN_TABLE_TO_COIN[table]
    asset_dir = coin.strip().lower()
    parquet_path = ONCHAIN_RAW_ROOT / asset_dir / f"{table}_validated.parquet"
    parquet_record = {
        "tipo": "validated_snapshot",
        "simbolo": coin,
        "archivo": parquet_path.name,
        "tamaño_mb": None,
        "filas": None,
        "columnas": None,
        "min_timestamp": None,
        "max_timestamp": None,
    }

    if parquet_path.exists():
        parquet_record["tamaño_mb"] = round(parquet_path.stat().st_size / (1024 * 1024), 2)
        try:
            parquet_df = pd.read_parquet(parquet_path)
            bucket_timestamps = pd.to_datetime(
                parquet_df["bucket_4h"], utc=True, errors="coerce"
            ).dropna()
            parquet_record.update(
                {
                    "filas": len(parquet_df),
                    "columnas": len(parquet_df.columns),
                    "min_timestamp": bucket_timestamps.min()
                    if not bucket_timestamps.empty
                    else None,
                    "max_timestamp": bucket_timestamps.max()
                    if not bucket_timestamps.empty
                    else None,
                }
            )
        except Exception:  # Conserva el registro aunque falle el análisis del `parquet`
            pass

    parquet_rows.append(parquet_record)

parquet_inventory_onchain_df = pd.DataFrame(parquet_rows, columns=ONCHAIN_PARQUET_COLUMNS)
archivos_parquet_presentes = int(parquet_inventory_onchain_df["tamaño_mb"].notna().sum())
archivos_parquet_esperados = ONCHAIN_PARQUET_EXPECTED_FILES
parquet_inventory_onchain_ok = archivos_parquet_presentes == archivos_parquet_esperados

print("Inventario de artefactos Parquet en disco")
display(parquet_inventory_onchain_df)
print(
    "Inventario de Parquet canónico: "
    f"{'OK' if parquet_inventory_onchain_ok else 'PENDIENTE'} "
    f"({archivos_parquet_presentes}/{archivos_parquet_esperados} ficheros)"
)


Inventario de artefactos Parquet en disco


,tipo,simbolo,archivo,tamaño_mb,filas,columnas,min_timestamp,max_timestamp
0,validated_snapshot,BTC,fact_onchain_btc_4h_validated.parquet,2.43,19131,27,2017-08-01 00:00:00+00:00,2026-04-24 08:00:00+00:00
1,validated_snapshot,ETH,fact_onchain_eth_4h_validated.parquet,3.06,19130,27,2017-08-01 00:00:00+00:00,2026-04-24 04:00:00+00:00
2,validated_snapshot,SOL,fact_onchain_sol_4h_validated.parquet,1.85,12174,26,2020-10-03 12:00:00+00:00,2026-04-24 08:00:00+00:00
3,validated_snapshot,XRP,fact_onchain_xrp_4h_validated.parquet,2.16,17458,26,2018-05-01 00:00:00+00:00,2026-04-18 12:00:00+00:00
4,validated_snapshot,BNB,fact_onchain_bnb_4h_validated.parquet,1.49,12369,26,2020-09-01 00:00:00+00:00,2026-04-24 08:00:00+00:00


Inventario de Parquet canónico: OK (5/5 ficheros)


## 5. Exportación de artefactos para reproducibilidad

Las tablas vistas son útiles dentro del cuaderno, pero la memoria del TFG y la auditoría externa necesitan **artefactos persistentes** en disco. En esta celda se vuelca todo lo anterior bajo `reports/validation/onchain/`, con un sello UTC (`<stamp>`) en el nombre para no sobrescribir corridas previas (p. ej. `20260603T235232Z` en esta corrida).

- **JSON consolidado:** `onchain_validation_<stamp>.json` (metadatos, *freeze*, resultado global, `checks`, cobertura, modos, huecos, duplicados, integridad, inventario *Parquet*, distribución de métricas).

- **Nueve CSV** (mismo `<stamp>`): `summary`, `cobertura`, `cobertura_objetivo`, `duplicados`, `integridad`, `parquet_inventory`, `distribucion_metrics`, `modos`, `ingestion_runs`. El CSV `gaps` solo se escribe si se detectan huecos estrictos (> 4h).

- **Figuras:** ninguna en este cuaderno (solo tablas y exportación tabular).

Al final se imprime el **resultado global** (`PASS`/`FAIL`) y un mini-checklist (cobertura, histórico, *live*, *repair*, huecos, duplicados, integridad, *Parquet*). Por último se libera la conexión con `engine.dispose()` si el kernel sigue vivo.

In [9]:
# Exporta JSON y CSV a disco bajo un solo `RUN_STAMP` y define con antelación el diccionario `export_paths`
ONCHAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_paths = {
    "json": ONCHAIN_OUTPUT_DIR / f"onchain_validation_{RUN_STAMP}.json",
    "summary": ONCHAIN_OUTPUT_DIR / f"onchain_validation_summary_{RUN_STAMP}.csv",
    "cobertura": ONCHAIN_OUTPUT_DIR / f"onchain_validation_cobertura_{RUN_STAMP}.csv",
    "cobertura_objetivo": ONCHAIN_OUTPUT_DIR / f"onchain_validation_cobertura_objetivo_{RUN_STAMP}.csv",
    "gaps": ONCHAIN_OUTPUT_DIR / f"onchain_validation_gaps_{RUN_STAMP}.csv",
    "duplicados": ONCHAIN_OUTPUT_DIR / f"onchain_validation_duplicados_{RUN_STAMP}.csv",
    "distribucion": ONCHAIN_OUTPUT_DIR / f"onchain_validation_distribucion_metrics_{RUN_STAMP}.csv",
    "modos": ONCHAIN_OUTPUT_DIR / f"onchain_validation_modos_{RUN_STAMP}.csv",
    "auditoria_runs": ONCHAIN_OUTPUT_DIR / f"onchain_validation_ingestion_runs_{RUN_STAMP}.csv",
    "integridad": ONCHAIN_OUTPUT_DIR / f"onchain_validation_integridad_{RUN_STAMP}.csv",
    "parquet_inventory": ONCHAIN_OUTPUT_DIR / f"onchain_validation_parquet_inventory_{RUN_STAMP}.csv",
}

historico_ok = bool(
    (not resumen_modos_onchain_df.empty) and (resumen_modos_onchain_df["modo"] == "histórico").any()
)
live_ok = bool(
    (not resumen_modos_onchain_df.empty) and (resumen_modos_onchain_df["modo"] == "live").any()
)
repair_ok = bool(
    (not resumen_modos_onchain_df.empty) and (resumen_modos_onchain_df["modo"] == "repair").any()
)
duplicados_ok = bool(
    not duplicados_pk_df.empty
    and len(duplicados_pk_df) == len(ONCHAIN_TABLES)
    and (duplicados_pk_df["duplicados_pk"] == 0).all()
)
gaps_ok = len(gaps_dfs) == 0
cobertura_ok = bool(
    not cobertura_objetivo_df.empty and cobertura_objetivo_df["tiene_datos"].all()
)

gaps_combinados_df = (
    pd.concat(
        [
            gaps_frame.assign(tabla=table, simbolo=ONCHAIN_TABLE_TO_COIN[table])
            for table, gaps_frame in gaps_dfs.items()
        ],
        ignore_index=True,
    )
    if gaps_dfs
    else pd.DataFrame()
)
gaps_detalle = gaps_combinados_df.to_dict(orient="records") if not gaps_combinados_df.empty else []

payload = {
    "metadata": run_meta,
    "fecha_freeze_política": ONCHAIN_FREEZE_COVERAGE_DATE,
    "bucket_freeze_utc": ONCHAIN_FREEZE_BUCKET.isoformat()
    if ONCHAIN_FREEZE_BUCKET is not None
    else None,
    "activos": list(ONCHAIN_ASSETS),
    "resultado_global": "PASS" if accepted else "FAIL",
    "fallos_accionables": actionable_failures,
    "checks": summary_df.to_dict(orient="records"),
    "cobertura": {
        "verificado": cobertura_ok,
        "detalle": evidencia_cobertura_vista_df.to_dict(orient="records")
        if not evidencia_cobertura_vista_df.empty
        else [],
        "objetivo_freeze": cobertura_objetivo_df.to_dict(orient="records")
        if not cobertura_objetivo_df.empty
        else [],
    },
    "modos": resumen_modos_onchain_df.to_dict(orient="records")
    if not resumen_modos_onchain_df.empty
    else [],
    "completitud_etl": {
        "historico_detectado": historico_ok,
        "live_detectado": live_ok,
        "repair_detectado": repair_ok,
    },
    "huecos_temporales": {
        "verificado": gaps_ok,
        "sin_huecos_estrictos": gaps_ok,
        "detalle": gaps_detalle,
    },
    "duplicados_pk": {
        "verificado": duplicados_ok,
        "cero_duplicados_todos_activos": duplicados_ok,
        "detalle": duplicados_pk_df.to_dict(orient="records")
        if not duplicados_pk_df.empty
        else [],
    },
    "integridad_imputaciones": {
        "verificado": integridad_onchain_ok,
        "detalle": integridad_onchain_df.to_dict(orient="records")
        if not integridad_onchain_df.empty
        else [],
    },
    "inventario_parquet": {
        "verificado": parquet_inventory_onchain_ok,
        "archivos_encontrados": archivos_parquet_presentes,
        "archivos_esperados": archivos_parquet_esperados,
        "detalle": parquet_inventory_onchain_df.to_dict(orient="records")
        if not parquet_inventory_onchain_df.empty
        else [],
    },
}

export_paths["json"].write_text(
    json.dumps(payload, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
summary_df.to_csv(export_paths["summary"], index=False)

if not evidencia_cobertura_vista_df.empty:
    evidencia_cobertura_vista_df.to_csv(export_paths["cobertura"], index=False)
if not cobertura_objetivo_df.empty:
    cobertura_objetivo_df.to_csv(export_paths["cobertura_objetivo"], index=False)
if not gaps_combinados_df.empty:
    gaps_combinados_df.to_csv(export_paths["gaps"], index=False)
if not duplicados_pk_df.empty:
    duplicados_pk_df.to_csv(export_paths["duplicados"], index=False)
if not integridad_onchain_df.empty:
    integridad_onchain_df.to_csv(export_paths["integridad"], index=False)
if not parquet_inventory_onchain_df.empty:
    parquet_inventory_onchain_df.to_csv(export_paths["parquet_inventory"], index=False)
if stats_dfs:
    distribution_frames: list[pd.DataFrame] = []
    for table, stats_frame in stats_dfs.items():
        distribution_frame = stats_frame.copy()
        distribution_frame.insert(0, "tabla", table)
        distribution_frame.insert(1, "simbolo", ONCHAIN_TABLE_TO_COIN[table])
        distribution_frames.append(distribution_frame)
    pd.concat(distribution_frames, ignore_index=True).to_csv(
        export_paths["distribucion"],
        index=False,
    )
if not resumen_modos_onchain_df.empty:
    resumen_modos_onchain_df.to_csv(export_paths["modos"], index=False)
if not evidencia_auditoria_onchain_df.empty:
    evidencia_auditoria_onchain_df.to_csv(export_paths["auditoria_runs"], index=False)

# Muestra un resumen mínimo en consola; los detalles quedan en disco
print(f"Resultado global: {'PASS' if accepted else 'FAIL'}")
print(f"Cobertura por activo: {'OK' if cobertura_ok else 'PENDIENTE'}")
print(f"Completitud histórico: {'OK' if historico_ok else 'PENDIENTE'}")
print(f"Completitud live: {'OK' if live_ok else 'PENDIENTE'}")
print(f"Completitud repair: {'OK' if repair_ok else 'PENDIENTE'}")
print(f"Huecos temporales: {'OK' if gaps_ok else 'DETECTADOS'}")
print(f"Duplicados = 0 verificado: {'OK' if duplicados_ok else 'PENDIENTE'}")
print(f"Integridad por activo: {'OK' if integridad_onchain_ok else 'PENDIENTE'}")
print(f"Inventario Parquet: {'OK' if parquet_inventory_onchain_ok else 'PENDIENTE'}")
print(f"Evidencia JSON: {export_paths['json']}")
print(f"Resumen de comprobaciones (CSV): {export_paths['summary']}")
if not evidencia_cobertura_vista_df.empty:
    print(f"Cobertura (CSV): {export_paths['cobertura']}")
if not cobertura_objetivo_df.empty:
    print(f"Cobertura objetivo (CSV): {export_paths['cobertura_objetivo']}")
if gaps_dfs:
    print(f"Huecos temporales (CSV): {export_paths['gaps']}")
if not duplicados_pk_df.empty:
    print(f"Duplicados de clave (CSV): {export_paths['duplicados']}")
if not integridad_onchain_df.empty:
    print(f"Integridad e imputaciones (CSV): {export_paths['integridad']}")
if not parquet_inventory_onchain_df.empty:
    print(f"Inventario de Parquet (CSV): {export_paths['parquet_inventory']}")
if stats_dfs:
    print(f"Distribución de métricas (CSV): {export_paths['distribucion']}")
if not resumen_modos_onchain_df.empty:
    print(f"Modos histórico, live y repair (CSV): {export_paths['modos']}")
if not evidencia_auditoria_onchain_df.empty:
    print(f"Auditoría de ingesta on-chain (CSV): {export_paths['auditoria_runs']}")


if engine:
    engine.dispose()  # Reintegra la conexión al pool al cierre de la exportación


Resultado global: PASS
Cobertura por activo: OK
Completitud histórico: OK
Completitud live: OK
Completitud repair: PENDIENTE
Huecos temporales: OK
Duplicados = 0 verificado: OK
Integridad por activo: OK
Inventario Parquet: OK
Evidencia JSON: /app/reports/validation/onchain/onchain_validation_20260604T000828Z.json
Resumen de comprobaciones (CSV): /app/reports/validation/onchain/onchain_validation_summary_20260604T000828Z.csv
Cobertura (CSV): /app/reports/validation/onchain/onchain_validation_cobertura_20260604T000828Z.csv
Cobertura objetivo (CSV): /app/reports/validation/onchain/onchain_validation_cobertura_objetivo_20260604T000828Z.csv
Duplicados de clave (CSV): /app/reports/validation/onchain/onchain_validation_duplicados_20260604T000828Z.csv
Integridad e imputaciones (CSV): /app/reports/validation/onchain/onchain_validation_integridad_20260604T000828Z.csv
Inventario de Parquet (CSV): /app/reports/validation/onchain/onchain_validation_parquet_inventory_20260604T000828Z.csv
Distribució

## 6. Conclusiones

Para el universo `{BTC, ETH, BNB, XRP, SOL}` en *timeframe* 4h, esta ejecución (`stamp` `20260603T235232Z`, `fecha_hora_utc` **2026-06-03 23:52 UTC**) deja fijada una lectura en tres capas:

1. **Validador:** resultado global `PASS`; las ocho comprobaciones en `PASS` (incluido `FREEZE_COVERAGE` hasta **2026-02-28 20:00 UTC**); **0** duplicados PK y **sin huecos** estrictos (> 4h) en las cinco facts.
2. **Calidad por activo:** **80,262** filas 4h agregadas en las cinco facts; BTC/ETH con históricos largos (**19,131** / **19,130** filas); XRP **17,458**; SOL/BNB con arranques más tardíos (**12,174** / **12,369** filas). Relleno de rejilla acotado: **18** filas en SOL (**99.85 %** real) y **1** en BNB (**99.99 %** real; `is_grid_padded`).
3. **Operativa y reproducibilidad:** cohorte de auditoría con **17** ejecuciones (**1** histórico, **16** *live*; **17,166** / **146,428** filas reportadas por modo; sin modo `repair` -> export marca completitud *repair* como `PENDIENTE`); inventario de **5/5** *Parquet* canónicos; JSON + nueve CSV bajo `reports/validation/onchain/`.

La capa *on-chain* puede integrarse en el backbone multifuente junto al resto de fuentes de ingesta.

**Nota.** Resultado global, métricas SQL, modos en `ingestion_runs` y rutas con `stamp` dependen del estado del proyecto al ejecutar el cuaderno; en otra corrida pueden cambiar.